In [1]:
import kagglehub

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from pytorch_tabnet.tab_model import TabNetClassifier

In [2]:
DATA_PATH = Path("/kaggle/input/competitions/playground-series-s6e5")
TARGET = "PitNextLap"

In [3]:
train_df = pd.read_csv(DATA_PATH / "train.csv")
test_df = pd.read_csv(DATA_PATH / "test.csv")

In [4]:
def preprocess_data(train_df,test_df,target_col,):
    X = train_df.drop(columns=[target_col]).copy()

    y = train_df[target_col].astype(int).values

    test_ids = test_df["id"]

    X = X.drop(columns=["id"])
    X_test = test_df.drop(columns=["id"]).copy()

    categorical_cols = (
        X.select_dtypes(
            include=["object", "category"]
        )
        .columns
        .tolist()
    )

    numerical_cols = (
        X.select_dtypes(
            exclude=["object", "category"]
        )
        .columns
        .tolist()
    )

    # Numerical preprocessing
    num_imputer = SimpleImputer(
        strategy="median"
    )

    scaler = StandardScaler()

    X[numerical_cols] = num_imputer.fit_transform(
        X[numerical_cols]
    )

    X_test[numerical_cols] = num_imputer.transform(
        X_test[numerical_cols]
    )

    X[numerical_cols] = scaler.fit_transform(
        X[numerical_cols]
    )

    X_test[numerical_cols] = scaler.transform(
        X_test[numerical_cols]
    )

    # Categorical preprocessing
    for col in categorical_cols:
        X[col] = X[col].fillna("missing")
        X_test[col] = X_test[col].fillna("missing")

        all_values = pd.concat(
            [
                X[col],
                X_test[col],
            ],
            axis=0,
        ).astype(str)

        mapping = {
            value: idx
            for idx, value in enumerate(
                all_values.unique()
            )
        }

        X[col] = (
            X[col]
            .astype(str)
            .map(mapping)
            .astype(np.int32)
        )

        X_test[col] = (
            X_test[col]
            .astype(str)
            .map(mapping)
            .astype(np.int32)
        )

    return (
        X.values.astype(np.float32),
        y,
        X_test.values.astype(np.float32),
        test_ids,
    )

In [5]:
(X,y,X_test,test_ids,) = preprocess_data(train_df,test_df,TARGET,)

In [6]:
X_train, X_valid, y_train, y_valid = train_test_split(X,y,test_size=0.10,random_state=42,stratify=y,)

In [7]:
tabnet = TabNetClassifier(
    device_name='cuda',
    n_d=32,
    n_a=32,
    n_steps=5,
    gamma=1.5,
    lambda_sparse=1e-4,
    optimizer_params=dict(lr=2e-2),
    mask_type="entmax",
)

/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cuda
  warnings.warn(f"Device used : {self.device}")


In [8]:
tabnet.fit(
    X_train=X_train,
    y_train=y_train,
    eval_set=[(X_valid, y_valid)],
    eval_name=["valid"],
    eval_metric=["accuracy"],
    max_epochs=25,
    patience=15,
    batch_size=4096,
    virtual_batch_size=512,
)

epoch 0  | loss: 0.36853 | valid_accuracy: 0.85285 |  0:00:07s
epoch 1  | loss: 0.3175  | valid_accuracy: 0.86622 |  0:00:14s
epoch 2  | loss: 0.29905 | valid_accuracy: 0.86799 |  0:00:20s
epoch 3  | loss: 0.29302 | valid_accuracy: 0.87075 |  0:00:27s
epoch 4  | loss: 0.28974 | valid_accuracy: 0.87309 |  0:00:34s
epoch 5  | loss: 0.2856  | valid_accuracy: 0.87485 |  0:00:40s
epoch 6  | loss: 0.28167 | valid_accuracy: 0.87544 |  0:00:47s
epoch 7  | loss: 0.27945 | valid_accuracy: 0.87751 |  0:00:54s
epoch 8  | loss: 0.27664 | valid_accuracy: 0.87612 |  0:01:00s
epoch 9  | loss: 0.27317 | valid_accuracy: 0.87972 |  0:01:07s
epoch 10 | loss: 0.26971 | valid_accuracy: 0.88234 |  0:01:14s
epoch 11 | loss: 0.27129 | valid_accuracy: 0.88054 |  0:01:20s
epoch 12 | loss: 0.26675 | valid_accuracy: 0.88316 |  0:01:27s
epoch 13 | loss: 0.26118 | valid_accuracy: 0.88377 |  0:01:33s
epoch 14 | loss: 0.25907 | valid_accuracy: 0.88386 |  0:01:40s
epoch 15 | loss: 0.25742 | valid_accuracy: 0.88462 |  0

/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


In [9]:
valid_preds = tabnet.predict(X_valid)

In [10]:
acc = accuracy_score(y_valid,valid_preds)
print(f"Validation Accuracy: {acc:.6f}")

Validation Accuracy: 0.887439


In [11]:
test_preds = tabnet.predict(X_test)

submission = pd.DataFrame(
    {
        "id": test_ids,
        TARGET: test_preds,
    }
)

submission.to_csv(
    "submission.csv",
    index=False,
)

print("submission.csv saved")

submission.csv saved
